In [ ]:
!pip install datasets transformers
!pip install accelerate

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import pipeline
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup

GBERT1: Finetuning Gbert-base with twitter-dataset

In [ ]:
twitter_dataset = load_dataset("Alienmaster/german_politicians_twitter_sentiment")

In [ ]:
print(twitter_dataset)

In [ ]:
train_data = twitter_dataset["train"]
test_data = twitter_dataset["test"]

In [ ]:
model_name = "deepset/gbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
#add new column "label" with numerical values for sentiment: 0:positive, 1:negative, 2:neutral
label_mapping = {1: 0, 2: 1, 3: 2}

train_data = train_data.map(lambda x: {"label": label_mapping[x["majority_sentiment"]]})
test_data = test_data.map(lambda x: {"label": label_mapping[x["majority_sentiment"]]})

In [ ]:
#delete unnecessary columns
train_data = train_data.remove_columns(["ID", "majority_sentiment"])
test_data = test_data.remove_columns(["ID", "majority_sentiment"])

In [ ]:
print(train_data[0])

In [ ]:
#Tokenize data 
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

In [ ]:
train_tokenized = train_data.map(tokenize_function, batched=True)
test_tokenized = test_data.map(tokenize_function, batched=True)

In [ ]:
#Convert to pytorch tensors for the model
train_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
num_labels = len(label_mapping)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

In [ ]:
#Defining training and evaluation parameters
def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels, preds, average="macro")
  acc = accuracy_score(labels, preds)
  precision = precision_score(labels, preds, average="macro")
  recall = recall_score(labels, preds, average="macro")
  return {"precision": precision, "recall": recall, "acc": acc, "f1": f1}

In [ ]:
batch_size = 8
logging_steps = len(train_tokenized) // batch_size
model_name = f"{model_name}-finetuned-twitter"
training_args = TrainingArguments(output_dir = model_name,
                                  num_train_epochs = 4,
                                  learning_rate = 2e-5,
                                  logging_dir=None,
                                  logging_strategy="no",
                                  report_to=None,
                                  per_device_train_batch_size = batch_size,
                                  per_device_eval_batch_size = batch_size,
                                  evaluation_strategy="no",
                                  disable_tqdm = False,
                                  logging_steps = logging_steps,
                                  log_level="info")

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset=train_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
import wandb
wandb.init(mode="disabled")

In [ ]:
trainer.train()

In [ ]:
#evaluation on testplit
results = trainer.evaluate(test_tokenized)

In [ ]:
results_df = pd.DataFrame([results])

In [ ]:
print(results_df)

In [ ]:
#save the model
model_path = "/content/gbert_finetuned_twitter"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

In [ ]:
import shutil
from google.colab import files

In [ ]:
shutil.make_archive("/content/gbert_finetuned_twitter", 'zip', "/content/gbert_finetuned_twitter")

In [ ]:
files.download("/content/gbert_finetuned_twitter.zip")